In [199]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

## Problem 7

In [200]:
ames = pd.read_csv('./data/ames_prices.csv')

# looking
print(ames['Central.Air'].value_counts(dropna=False))

Central.Air
Y    2734
N     196
Name: count, dtype: int64


In [201]:
# building indicator
central_air = (ames['Central.Air'] == 'Y').astype(int).to_numpy()
n = len(central_air)
print(n)


2930


In [202]:
# compute estimate
p_hat = central_air.mean()
print(p_hat)

0.9331058020477816


In [203]:
# bootstrapping
seed = 2026
R = 2000
rng = np.random.default_rng(seed)

boot_p = np.empty(R)
for r in range(R):
    sample = rng.choice(central_air, size=n, replace=True)
    boot_p[r] = sample.mean()

In [204]:
# computing stats
# standard error of bootstrap
se = np.std(boot_p, ddof=1)
print(se)

0.004482493370050803


In [205]:
z_vals = {90: 1.645, 95: 1.96, 99:2.576}

for conf, z_val in z_vals.items():
    lo = p_hat - z_val * se
    hi = p_hat + z_val * se
    print(f"{conf}% CI: ({lo:.4f}, {hi:.4f})")



90% CI: (0.9257, 0.9405)
95% CI: (0.9243, 0.9419)
99% CI: (0.9216, 0.9447)


In [206]:
# test null hypothesis 
from scipy.stats import norm
theta0 = 0.90
z = (p_hat - theta0) / se
p_value = 2 * norm.cdf(-abs(z))
print(f"z = {z:.3f}, p = {p_value:.4g}") # .4g does scientific notation

z = 7.386, p = 1.518e-13


In [207]:
for conf, z_val in z_vals.items():
    alpha = 1 - conf/100
    lo = p_hat - z_val * se
    hi = p_hat + z_val * se

    reject = p_value < alpha
    contains = lo <= theta0 <= hi

    print(f"Alpha = {alpha:.2f}: reject = {reject}")
    print(f"CI contains 0.90 = {contains}, agree = {reject != contains}")


Alpha = 0.10: reject = True
CI contains 0.90 = False, agree = True
Alpha = 0.05: reject = True
CI contains 0.90 = False, agree = True
Alpha = 0.01: reject = True
CI contains 0.90 = False, agree = True


# leaving room for interpretation

# 8

In [208]:
print(ames['price'].dtype, ames['area'].dtype)


int64 int64


In [209]:
x = np.arcsinh(ames['area'].to_numpy())
y = np.arcsinh(ames['price'].to_numpy())
n = len(x)
rho_hat = np.corrcoef(x, y)[0,1] # np.corrcoef gives 2x2 matrix. [0, 1] entry is corr between x and y.


In [210]:
# paired bootstrap
rng = np.random.default_rng(seed)
boot_rho = np.empty(R)
for r in range(R):
    idx = rng.integers(0, n, size=n) # pick n row numbers with replacement
    boot_rho[r] = np.corrcoef(x[idx], y[idx])[0,1]

If we sampled separately, then each bootstrap house would get a random area of one house and a random price from a different house. That breaks the link between area and price, so the bootstrap correlations would cluster around 0 regardless of the true relationship.

In [211]:
# computing stats
# standard error of bootstrap
se = np.std(boot_rho, ddof=1)
print(se)

0.009829548364063145


In [212]:
z_vals = {90: 1.645, 95: 1.96, 99: 2.576}
for conf, z_val in z_vals.items():
    lo = rho_hat - z_val * se
    hi = rho_hat + z_val * se
    print(f"{conf}% CI: ({lo:.4f}, {hi:.4f})")

90% CI: (0.7070, 0.7393)
95% CI: (0.7039, 0.7424)
99% CI: (0.6978, 0.7485)


In [213]:
# test null hypothesis 
theta0 = 0
z = (rho_hat - theta0) / se
p_value = 2 * norm.cdf(-abs(z))
print(f"z = {z:.3f}, p = {p_value:.4g}") # .4g does scientific notation

z = 73.571, p = 0


In [214]:
for conf, z_val in z_vals.items():
    alpha = 1 - conf/100
    lo = rho_hat - z_val * se
    hi = rho_hat + z_val * se

    reject = p_value < alpha
    contains = lo <= theta0 <= hi

    print(f"Alpha = {alpha:.2f}: reject = {reject}")
    print(f"CI contains 0 = {contains}, agree = {reject != contains}")

Alpha = 0.10: reject = True
CI contains 0 = False, agree = True
Alpha = 0.05: reject = True
CI contains 0 = False, agree = True
Alpha = 0.01: reject = True
CI contains 0 = False, agree = True


# room for text answer

# 9

In [215]:
health_df = pd.read_csv("./data/metabric.csv")
health_df.head()

,Age at Diagnosis,Type of Breast Surgery,Cancer Type,Chemotherapy,Hormone Therapy,Lymph nodes examined positive,Mutation Count,Nottingham prognostic index,Overall Survival (Months),Overall Survival Status,Radio Therapy,TMB (nonsynonymous),Tumor Size,Tumor Stage
0,43.19,BREAST CONSERVING,Breast Cancer,NO,YES,0.0,2.0,4.020,84.633333,0:LIVING,YES,2.615035,10.0,1.0
1,48.87,MASTECTOMY,Breast Cancer,YES,YES,1.0,2.0,4.030,163.700000,1:DECEASED,NO,2.615035,15.0,2.0
2,47.68,MASTECTOMY,Breast Cancer,YES,YES,3.0,1.0,4.050,164.933333,0:LIVING,YES,1.307518,25.0,2.0
3,76.97,MASTECTOMY,Breast Cancer,YES,YES,8.0,2.0,6.080,41.366667,1:DECEASED,YES,2.615035,40.0,2.0
4,78.77,MASTECTOMY,Breast Cancer,NO,YES,0.0,4.0,4.062,7.800000,1:DECEASED,YES,5.230071,31.0,4.0


In [216]:
survival_months = health_df['Overall Survival (Months)']
n = len(health_df['Overall Survival (Months)'])
sample_mean = health_df['Overall Survival (Months)'].mean()


In [217]:
# bootstrapping
seed = 2026
R = 2000
rng = np.random.default_rng(seed)

boot_mean = np.empty(R)
for r in range(R):
    sample = rng.choice(survival_months, size=n, replace=True)
    boot_mean[r] = sample.mean()

In [218]:
# computing stats
# standard error of bootstrap
se = np.std(boot_mean, ddof=1)
print(se)

2.1467076969725065


In [219]:
z_vals = {90: 1.645, 95: 1.96, 99:2.576}

for conf, z_val in z_vals.items():
    lo = sample_mean - z_val * se
    hi = sample_mean + z_val * se
    print(f"{conf}% CI: ({lo:.4f}, {hi:.4f})")


90% CI: (125.5630, 132.6256)
95% CI: (124.8867, 133.3018)
99% CI: (123.5644, 134.6242)


In [220]:
# test null hypothesis 
from scipy.stats import norm
theta0 = 120
z = (sample_mean - theta0) / se
p_value = 2 * norm.cdf(-abs(z))
print(f"z = {z:.3f}, p = {p_value:.4g}") # .4g does scientific notation

z = 4.236, p = 2.271e-05


In [221]:
for conf, z_val in z_vals.items():
    alpha = 1 - conf/100
    lo = sample_mean - z_val * se
    hi = sample_mean + z_val * se

    reject = p_value < alpha
    contains = lo <= theta0 <= hi

    print(f"Alpha = {alpha:.2f}: reject = {reject}\n")
    print(f"CI contains 120 = {contains}, agree = {reject != contains}\n")

Alpha = 0.10: reject = True

CI contains 120 = False, agree = True

Alpha = 0.05: reject = True

CI contains 120 = False, agree = True

Alpha = 0.01: reject = True

CI contains 120 = False, agree = True



interpretation

# 10

In [ ]:

health_df['deceased'] = (health_df['Overall Survival Status'] == "1:DECEASED").astype(int)
health_df = health_df.dropna(subset=['Overall Survival Status', 'Chemotherapy'])

In [ ]:
yes = health_df.loc[health_df['Chemotherapy'] == "YES", "deceased"].to_numpy()
no = health_df.loc[health_df['Chemotherapy'] == "NO", 'deceased'].to_numpy()
n_yes, n_no = len(yes), len(no)

p_yes = yes.mean()
p_no = no.mean()
delta_hat = p_yes - p_no

print(n_yes, n_no)
print(p_yes, p_no, delta_hat)

# 11

In [222]:
nhanes = pd.read_csv('./data/nhanes_data_17_18.csv')


C:\Users\Evant\AppData\Local\Temp\ipykernel_21900\1912704617.py:1: DtypeWarning: Columns (142: ServedActiveDutyInUsArmedForces) have mixed types. Specify dtype option on import or set low_memory=False.
  nhanes = pd.read_csv('./data/nhanes_data_17_18.csv')


In [223]:
adults = nhanes.loc[(nhanes['AgeInYearsAtScreening'] >= 18) & 
           (nhanes['SleepHoursWeekdaysOrWorkdays'].notna())]
sleep = adults['SleepHoursWeekdaysOrWorkdays'].to_numpy()
n = len(sleep)

In [224]:

sleep_mean = adults['SleepHoursWeekdaysOrWorkdays'].mean()
print(sleep_mean, n)


7.62124248496994 5489


In [225]:
# bootstrapping
seed = 2026
R = 2000
rng = np.random.default_rng(seed)

boot_sleep = np.empty(R)
for r in range(R):
    sample = rng.choice(sleep, size=n, replace=True)
    boot_sleep[r] = sample.mean()

In [226]:
# computing stats
# standard error of bootstrap
se = np.std(boot_sleep, ddof=1)
print(se)

0.02263818179692323


In [227]:
z_vals = {90: 1.645, 95: 1.96, 99:2.576}

for conf, z_val in z_vals.items():
    lo = sleep_mean - z_val * se
    hi = sleep_mean + z_val * se
    print(f"{conf}% CI: ({lo:.4f}, {hi:.4f})")


90% CI: (7.5840, 7.6585)
95% CI: (7.5769, 7.6656)
99% CI: (7.5629, 7.6796)


In [228]:
# test null hypothesis 
from scipy.stats import norm
theta0 = 8
z = (sleep_mean - theta0) / se
p_value = 2 * norm.cdf(-abs(z))
print(f"z = {z:.3f}, p = {p_value:.4g}") # .4g does scientific notation

z = -16.731, p = 7.803e-63


In [229]:
for conf, z_val in z_vals.items():
    alpha = 1 - conf/100
    lo = sleep_mean - z_val * se
    hi = sleep_mean + z_val * se

    reject = p_value < alpha
    contains = lo <= theta0 <= hi

    print(f"Alpha = {alpha:.2f}: reject = {reject}")
    print(f"CI contains 8 = {contains}, agree = {reject != contains}")

Alpha = 0.10: reject = True
CI contains 8 = False, agree = True
Alpha = 0.05: reject = True
CI contains 8 = False, agree = True
Alpha = 0.01: reject = True
CI contains 8 = False, agree = True
